# 00 - Connect, and list what the server holds

Can I open a session with an ETP server, and see its dataspaces and the resources in one of them? Can I tell when I asked wrongly?

`retp` is the Python bridge of the Rust ETP v1.2 client. A session is a context manager; a failure is an exception of `retp.errors`, never an empty result that means "not supported".

## Before you run

Settings come from environment variables only. No server name, user or token is written in this notebook, and the token is never printed.

| Variable | Needed | Meaning |
|---|---|---|
| `RETP_URL` | yes | `wss://...` ETP endpoint |
| `RETP_TOKEN` | yes | bearer token, without its scheme |
| `RETP_INSECURE` | no | `1` skips the certificate check (default: checked) |
| `RETP_PARTITION` | no | OSDU data partition |
| `RETP_DATASPACE` | no | the dataspace to read; when unset, the first one listed is taken and the notebook says so |
| `RETP_WRITE_DATASPACE` | no | a dataspace you created for the test and will delete; only the writing notebook uses it, and it does nothing without it |
| `RETP_EPC` | no | path of an EPC to push (its `.h5` beside it) |

When a precondition is missing, a cell prints why and the rest of the notebook is skipped without an error. When the server answers something the ETP 1.2 specification forbids, an `assert` fails with a message. The assertions check the **shape** of the answers, never the values a given server holds.

In [ ]:
import retp
import _server

KW = _server.connect_arguments()   # None, with a printed reason, when RETP_URL or RETP_TOKEN is missing
READY = KW is not None

## Opening a session

`retp.connect(url, token, ...)` opens the session; leaving the `with` block closes it. The token is handed to the Rust client and is in no `repr`, no attribute and no message. `session.warnings` says what the library had to say about the connection (plain text, unchecked certificate); `ping()` returns the round trip in seconds.

In [ ]:
import re

UUID = re.compile(r"[0-9a-fA-F]{8}-(?:[0-9a-fA-F]{4}-){3}[0-9a-fA-F]{12}")
TYPE = re.compile(r"/([a-z]+[0-9]+)\.([A-Za-z0-9_]+)\(")


def uuid_of(uri):
    found = UUID.findall(uri)
    return found[-1].lower() if found else None


def family(uri):
    found = TYPE.search(uri)
    return found.group(1) if found else None


def qualified_type(uri):
    found = TYPE.search(uri)
    return f"{found.group(1)}.{found.group(2)}" if found else None


def refused(call, *expected):
    # Run call(); it must raise one of expected. Anything else is a failure.
    try:
        call()
    except expected as error:
        return error
    except Exception as other:
        raise AssertionError(f"expected {[e.__name__ for e in expected]}, got {type(other).__name__}: {other}")
    raise AssertionError("the call returned instead of refusing")

In [ ]:
DATASPACE = None
if READY:
    with retp.connect(**KW) as session:
        assert session.writing is False, "a session opened without writing=True must say it is read-only"
        assert KW["token"] not in repr(session) and KW["token"] not in session.url, "the token leaked into the session"
        assert isinstance(session.warnings, list)
        for warning in session.warnings:
            print("warning:", warning)

        round_trip = session.ping()
        assert isinstance(round_trip, float) and round_trip >= 0, f"ping answered {round_trip!r}"
        print(f"ping: {round_trip * 1000:.1f} ms")

        spaces = session.dataspaces()
        print(len(spaces), "dataspace(s)")
        for row in spaces:
            assert {"uri", "path", "store_last_write", "store_created"} <= set(row), f"a dataspace row has the keys {sorted(row)}"
            assert row["uri"].startswith("eml:///"), f"{row['uri']!r} is not an ETP URI"
            print(" ", row["path"] or "(default)", row["uri"])
        DATASPACE = _server.dataspace_to_read(session)

## Resources of one dataspace

`resources(dataspace, *, depth, scope, types, edges, count)` is ETP's GetResources. `dataspace` is a path or a URI. `scope` is one of `self`, `sources`, `targets`, `sourcesOrSelf`, `targetsOrSelf`. `types` are qualified types; the bridge's user guide gives `resqml22.*` as an example of a wildcard. Here the family to ask for is taken from what the server itself listed, so the cell does not guess what the server holds.

In [ ]:
URIS = []
if READY and DATASPACE is not None:
    with retp.connect(**KW) as session:
        listed = session.resources(DATASPACE)
        assert set(listed) == {"resources", "edges"}, f"the listing has the keys {sorted(listed)}"
        rows = listed["resources"]
        for row in rows:
            assert {"uri", "name", "source_count", "target_count", "last_changed"} <= set(row), sorted(row)
            assert row["uri"].startswith("eml:///"), f"{row['uri']!r} is not an ETP URI"
        URIS = [row["uri"] for row in rows]
        assert len(URIS) == len(set(URIS)), "a resource is listed twice"
        print(len(rows), "resource(s) in", DATASPACE)
        for row in rows[:5]:
            print(" ", qualified_type(row["uri"]), "|", row["name"])

        if not URIS:
            print("The dataspace is empty: the filters below are skipped.")

In [ ]:
if URIS:
    with retp.connect(**KW) as session:
        # A wildcard: every type of one family, the family being one the server listed.
        families = sorted({family(uri) for uri in URIS if family(uri)})
        if families:
            wanted = families[0] + ".*"
            narrowed = session.resources(DATASPACE, types=[wanted])["resources"]
            expected = {uri for uri in URIS if family(uri) == families[0]}
            assert {row["uri"] for row in narrowed} == expected, f"types=[{wanted!r}] did not select the family"
            print(wanted, "->", len(narrowed), "of", len(URIS))
        else:
            print("TODO(verify): no URI carried a qualified type in the form family.Type; the wildcard was not tried.")

        # One exact type.
        exact = qualified_type(URIS[0])
        if exact:
            same = session.resources(DATASPACE, types=[exact])["resources"]
            assert same, f"types=[{exact!r}] lost the object that has this type"
            assert all(qualified_type(row["uri"]) == exact for row in same), f"types=[{exact!r}] returned another type"
            print(exact, "->", len(same))

        # Depth, edges, counts.
        deeper = session.resources(DATASPACE, depth=2)
        print("depth=2 ->", len(deeper["resources"]))

        with_edges = session.resources(DATASPACE, edges=True)
        for edge in with_edges["edges"]:
            assert {"source_uri", "target_uri", "relationship_kind"} <= set(edge), sorted(edge)
            assert edge["source_uri"].startswith("eml:///") and edge["target_uri"].startswith("eml:///")
        print("edges=True ->", len(with_edges["edges"]), "edge(s)")

        counted = session.resources(DATASPACE, count=True)
        for row in counted["resources"]:
            for key in ("source_count", "target_count"):
                assert row[key] is None or (isinstance(row[key], int) and row[key] >= 0), f"{key}={row[key]!r}"
        print("count=True ->", len(counted["resources"]), "row(s)")

        # The scope `self` on one object answers that object.
        alone = session.resources(URIS[0], scope="self")["resources"]
        assert uuid_of(URIS[0]) in {uuid_of(row["uri"]) for row in alone}, "scope='self' did not answer the object itself"
        print("scope='self' ->", len(alone), "resource(s)")

## A wrong request is refused by name

A wrong argument is a plain `ValueError`, raised before anything is sent. A case the bridge does not support raises `retp.Unsupported` and names it. Every other failure is a `retp.RetpError` subclass, and the tree is fixed: `ConnectionLost` is above `OutcomeUnknown`, and so on (see `retp.errors`).

In [ ]:
# These refusals happen before any connection: they run without a server.
assert issubclass(retp.Unsupported, retp.RetpError)
assert issubclass(retp.OutcomeUnknown, retp.ConnectionLost) and issubclass(retp.ConnectionLost, retp.RetpError)
assert not issubclass(ValueError, retp.RetpError)

error = refused(lambda: retp.connect("https://etp.example.org/", token="x"), retp.Unsupported)
assert "ws://" in str(error) or "wss://" in str(error), f"the refusal does not name the rule: {error}"
print("https URL ->", type(error).__name__)

error = refused(lambda: retp.connect("wss://etp.example.org/", at_once=0), ValueError)
assert "at_once" in str(error), f"the refusal does not name the argument: {error}"
print("at_once=0 ->", type(error).__name__)

In [ ]:
if URIS or (READY and DATASPACE is not None):
    with retp.connect(**KW) as session:
        error = refused(lambda: session.resources(DATASPACE, scope="sideways"), ValueError)
        assert "sideways" in str(error) and "scope" in str(error), f"the refusal does not name the scope: {error}"
        print("scope='sideways' ->", type(error).__name__, "|", error)

        error = refused(lambda: session.resources(DATASPACE, depth=0), ValueError)
        assert "depth" in str(error), f"the refusal does not name the argument: {error}"
        print("depth=0 ->", type(error).__name__, "|", error)
print("done")